In [46]:
! pip install openai


Looking in indexes: https://pypi.org/simple, https://nexus.margobank.net/repository/pypi/simple

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import random
import matplotlib.pyplot as plt
import logging
from openai import OpenAI
import enum
import base64
import asyncio

In [60]:
client = OpenAI()

In [62]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


In [ ]:
class GameStage(enum.Enum):
    first_choice = 1
    advised_choice = 2
    change_neighbors = 3

class HumanAgent:

    def __init__(self, client, id, preprompt):
        self.id = id
        self.client = client
        self.system_message = preprompt
        self.discussion = [ 
            {
                "role": "system",
                "content": [{ "type": "text", "text": preprompt}]
            }
        ]
        self.cumulative_score = 0

    async def get_answer(self, temp=0.7, model="gpt-4o"):

        if self.discussion[-1]["role"] == "user":
            completion = self.client.chat.completions.create(
                model=model,
                messages= self.discussion,
                temperature=temp,
            )
            
            self.discussion.append(
                {
                    "role": "assistant",
                    "content": [{ "type": "text", "text": completion.choices[0].message}],
                }
            )
            answer = completion.choices[0].message
        else:
            answer = self.discussion[-1]["content"][0]["text"]
        return answer, self.id
    
    def add_image_to_discussion(self, image_path: str):
        base64_image = encode_image(image_path[1:])    
        self.discussion.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", "text": "Here is the picture, What is your answer (give a value between 0 and 100) ?"},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                    },
                ],
            }
        )

    def add_feed_back_to_discussion(self, feedback: str):
        self.discussion.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", "text": feedback
                    }
                ],
            }
        )

In [52]:
#agent = HumanAgent(client, preprompt)
#agent.add_image_to_discussion("scatter_plt/image2.png")

#print(await agent.get_answer())

In [ ]:
from h11 import Response


class experimentGroup:
    def __init__(self, nb_rounds, nb_agents, jsonPath):
        self.nb_agents = nb_agents
        self.agents = []
        self.nb_rounds = nb_rounds
        self.images = json.load(open(jsonPath))
        self.game_stage = GameStage.first_choice
        self.agent_response = {}

    def generateAgent(self, preprompt, persona_prompts):
        if len(persona_prompts) == self.nb_agents:
            self.agents = [HumanAgent(client, id = i, preprompt= preprompt + persona_prompts[i]) for i in range(len(persona_prompts))]
        else:
            raise ValueError("chelou")
        
    async def run_first_stage(self, image_index, difficulty="low"):
        print("Game stage", GameStage.first_choice)
        for agent in self.agents:
            agent.add_image_to_discussion(self.images[image_index]["difficultyPath"][difficulty])
            response = agent.get_answer()
        for agent,response in await asyncio.gather(*response):
            self.agent_response[agent.id] = response

    async def run_experiment(self):
        for i in range(self.nb_rounds):
            self.run_first_stage()


            print("Game stage :", GameStage.advised_choice)
            for agent in self.agents:
                neighbor_responses = [f"- Participant {neighbor_id} : {self.agent_response[neighbor_id]}" for neighbor_id in agent.neighbor]
                agent.add_feed_back_to_discussion("Thank you for your answer, here is the answer of the participant you chose to follow :\n" \
                                                  "\n".join(neighbor_responses))

            
            for agent in self.agents:
                agent.add_image_to_discussion(self.images[i]["difficultyPath"]["medium"])
                response = await agent.get_answer()
                self.agent_responses.append(response)
                agent.add_feed_back_to_discussion("Thank you for your answer")
                print(response)
            self.game_stage = GameStage.change_neighbors


    @staticmethod
    def generate_scatter_plot(corellation_value, nb_points , image_path, plt=plt):
        """ generate scatter plot with a given corellation along a random linear line and return it"""
        x = [random.random() for _ in range(nb_points)]
        y = [corellation_value * x[i] + random.random() for i in range(nb_points)]
        plt.scatter(x, y)
        plt.savefig(f"scatter_plt/{image_path}.png")
        plt.close()


    def generate_images(self, correlations):
        """ generate images for the experiment"""
        image_data = []
        for i in range(len(correlations)):
            correlation = random.random()
            index = i%10 + i//10 * 30
            image = {
                "_id": i,
                "correct_answer": correlation,
                "difficultyPath": {
                    "high": f"scatter_plt/image_{index}.png",
                    "medium": f"scatter_plt/image_{index+10}.png",
                    "low": f"scatter_plt/image_{index+20}.png"
                }
            }
            image_data.append(image)
            experimentGroup.generate_scatter_plot(correlations[i], nb_points=20, image_path= f"image_{index}")
            experimentGroup.generate_scatter_plot(correlations[i], nb_points=40, image_path= f"image_{index + 10}")
            experimentGroup.generate_scatter_plot(correlations[i], nb_points=60, image_path= f"image_{index + 20}")
        return image_data
    
    
    def _set_neigbors_randomly(self):
        """ set 3 noeigbors randomly for each agent"""
        for agent in self.agents:
            agent.neighbor = [agent.id for agent in random.sample(self.agents, 3)]
    
    def add_first_stage_feedback(self):
        pass

    def get_image_recap(self):
        pass

In [ ]:
experiment = experimentGroup(nb_rounds= 10, nb_agents= 3, jsonPath= "task/recap.json")

persona_prompts = [""] * 3 

print(persona_prompts)
experiment.generateAgent(preprompt,persona_prompts)
experiment._set_neigbors_randomly()

first_stage_result = []
for agent in experiment.agents:
    agent.add_image_to_discussion(experiment.images[2]["difficultyPath"]["hard"])
    first_stage_result.append(await agent.get_answer(0.99))

first_stage_result

['', '', '']
3 3


[ChatCompletionMessage(content='0.9', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None),
 ChatCompletionMessage(content='0.85', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None),
 ChatCompletionMessage(content='0.95', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)]

In [81]:
experiment.images[10]

{'_id': 10,
 'correctAnswer': 0.07,
 'difficultyPath': {'easy': '/task/tasks/30.png',
  'hard': '/task/tasks/50.png',
  'medium': '/task/tasks/40.png'}}